# 03 - Optional: Campaign Experiment Planning

Does contact increase returns more in the next score decile than in the top decile?

This extension uses notebook 02's validation scores to plan a randomized experiment. Contact effects and outcomes are simulated; historical return rates are not verified no-contact probabilities. The dataset does not identify which visitors are reachable.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

project_dir = Path.cwd()
if not (project_dir / "src" / "data_pipeline.py").exists():
    project_dir = project_dir.parent
sys.path.insert(0, str(project_dir))

from src.provenance import load_validation_scores
from src.experiment import (
    GROUPS, score_groups, simulate_experiments, compare_groups, interaction_sample_size,
)


def format_pp(value):
    return f"{value * 100:.2f} pp"


scores, metadata = load_validation_scores(
    project_dir / "data/processed/validation_scores.parquet",
)
scores = score_groups(scores)
reference = scores.groupby("score range")["returned"].agg(["size", "mean"]).reindex(GROUPS)
reference.columns = ["visitors", "observed return rate"]
VISITORS = reference["visitors"].to_numpy(dtype=int)
REFERENCE_RETURN_RATES = reference["observed return rate"].to_numpy()
reference["contacted visitors"] = VISITORS // 2
reference["control visitors"] = VISITORS - VISITORS // 2
print(f"Ranking: {metadata['ranking']}; visitors: {len(scores):,}")
reference.style.format({"observed return rate": "{:.2%}"})

Ranking: XGBoost / full / max_depth=4; visitors: 46,854


,visitors,observed return rate,contacted visitors,control visitors
score range,,,,
Top 10%,4685,11.06%,2342,2343
Next 10%,4685,3.46%,2342,2343
Remaining 80%,37484,2.71%,18742,18742


## Assumptions and allocation

Form the top 10%, next 10%, and remaining 80% using rounded-down counts and seeded random tie-breaking, independent of returns. Contact half of each group; an odd-sized group has one extra control.

Visitors are independent, probabilities are fixed within groups, follow-up is complete, and contact has no spillover. Binomial arm totals give the same distribution as generating individual outcomes. The scenarios below check recovery of known effects.

In [2]:
EFFECTS = {
    "No effect": np.array([0.0, 0.0, 0.0]),
    "Equal effect": np.array([0.01, 0.01, 0.01]),
    "Next 10% benefits most": np.array([0.005, 0.03, 0.0]),
}
pd.DataFrame(EFFECTS, index=GROUPS).style.format(format_pp)

,No effect,Equal effect,Next 10% benefits most
Top 10%,0.00 pp,1.00 pp,0.50 pp
Next 10%,0.00 pp,1.00 pp,3.00 pp
Remaining 80%,0.00 pp,1.00 pp,0.00 pp


## Primary comparison: next-minus-top effect

Subtract the top group's contact effect from the next group's effect and add their variances. Positive values favor the next group. Use a two-sided 5% test for this primary comparison; group intervals are descriptive. A positive result in one group and an inconclusive result in another do not establish a difference.

## Repeat 500 times

Use the same simulator to measure estimation bias, group-interval coverage, and primary-comparison detection. Under zero interaction, exclusion of zero is a false positive; under a nonzero interaction, it measures detection probability. With 500 repeats, an estimated 95% coverage rate has about 1 percentage point of Monte Carlo standard error.

In [3]:
REPEATS = 500
repeated_rows, interaction_rows = [], []
for number, (scenario, effects) in enumerate(EFFECTS.items()):
    result = simulate_experiments(
        REFERENCE_RETURN_RATES, VISITORS, effects, repeats=REPEATS, seed=123 + number,
    )
    for group in GROUPS:
        values = result.loc[result["score range"] == group]
        truth = values["assumed true effect"].iloc[0]
        repeated_rows.append({
            "scenario": scenario, "score range": group,
            "assumed true effect": truth,
            "mean estimated effect": values["estimated effect"].mean(),
            "estimate standard deviation": values["estimated effect"].std(),
            "interval coverage": ((values["lower 95% bound"] <= truth)
                                  & (truth <= values["upper 95% bound"])).mean(),
        })
    interaction = compare_groups(result)
    truth = interaction["assumed true difference"].iloc[0]
    interaction_rows.append({
        "scenario": scenario,
        "assumed true difference": truth,
        "mean estimated difference": interaction["estimated next-minus-top effect"].mean(),
        "interval coverage": ((interaction["lower 95% bound"] <= truth)
                              & (truth <= interaction["upper 95% bound"])).mean(),
        "interval excludes zero": ((interaction["lower 95% bound"] > 0)
                                   | (interaction["upper 95% bound"] < 0)).mean(),
    })

pd.DataFrame(repeated_rows).set_index(["scenario", "score range"]).style.format({
    "assumed true effect": format_pp, "mean estimated effect": format_pp,
    "estimate standard deviation": format_pp, "interval coverage": "{:.1%}",
})

In [4]:
pd.DataFrame(interaction_rows).set_index("scenario").style.format({
    "assumed true difference": format_pp, "mean estimated difference": format_pp,
    "interval coverage": "{:.1%}", "interval excludes zero": "{:.1%}",
})

,assumed true difference,mean estimated difference,interval coverage,interval excludes zero
scenario,,,,
No effect,0.00 pp,0.01 pp,95.6%,4.4%
Equal effect,0.00 pp,0.08 pp,95.8%,4.2%
Next 10% benefits most,2.50 pp,2.54 pp,94.4%,60.6%


## Plan for a smaller, worthwhile difference

Use an illustrative next-minus-top difference of **1 percentage point**, with 80% power and a two-sided 5% significance level.

Assume zero effect in the top group and +1 point in the next group. The normal approximation sums four arm variances using the reference rates and equal arm sizes. Compare the required counts with those available; filling each decile requires ten times its size in eligible scored visitors.

In [5]:
MIN_DIFFERENCE = 0.01
plan = interaction_sample_size(*REFERENCE_RETURN_RATES[:2], difference=MIN_DIFFERENCE)
planning = pd.DataFrame({
    "available visitors": VISITORS[:2],
    "required visitors": plan["per score group"],
}, index=GROUPS[:2])
print(f"80% power target: {plan['total top and next']:,} visitors across the two groups")
print(f"Eligible scored population needed at 10% per group: {plan['per score group'] * 10:,}")
planning

80% power target: 85,604 visitors across the two groups
Eligible scored population needed at 10% per group: 428,020


,available visitors,required visitors
Top 10%,4685,42802
Next 10%,4685,42802


In [6]:
# Check the approximation under exactly the planning assumptions.
planned_counts = np.array([plan["per score group"], plan["per score group"], VISITORS[2]])
planned = simulate_experiments(
    REFERENCE_RETURN_RATES, planned_counts, np.array([0.0, MIN_DIFFERENCE, 0.0]),
    repeats=REPEATS, seed=321,
)
planned_comparison = compare_groups(planned)
detected = (planned_comparison["lower 95% bound"] > 0).mean()
print(f"Simulated positive detection at the planned size: {detected:.1%} ({REPEATS} repeats)")

Simulated positive detection at the planned size: 81.8% (500 repeats)


## What this changes

Before running a campaign, fix the outreach ranking and decile boundaries, recruit enough reachable visitors, randomize contact within each group, and analyze all assigned visitors after follow-up.